# B7n · Nitrosamine carcinogenic potency: a regression model you can interrogate
**QSAR primer notebook** &nbsp;|&nbsp; companion to the CPDB TD$_{50}$ compilation (the spreadsheet and the LaTeX report)

N-nitrosamines are a class of mutagenic impurities that regulators ask drug makers to control at very low intakes. Their rodent carcinogenic potency spans four orders of magnitude, and it is governed by one piece of chemistry: a nitrosamine is not itself reactive; a cytochrome P450 enzyme must first remove a hydrogen from a carbon next to the nitrosamine nitrogen (**$\alpha$-hydroxylation**), after which the molecule falls apart into a diazonium ion that alkylates DNA. Few $\alpha$-hydrogens, bulky $\alpha$-carbons, or electron-withdrawing groups next to the $\alpha$-carbon slow that step down. A model that encodes this should beat a model that only counts atoms.

This notebook builds a **regression model of rat TD$_{50}$** for N-nitrosamines from the Carcinogenic Potency Database (CPDB; Gold et al., *Environ. Health Perspect.* 1984; 58:9-319), and does it the careful way:

* the target is the **molar potency** $\mathrm{pTD}_{50}=-\log_{10}(\mathrm{TD}_{50}/\mathrm{mol\;kg^{-1}\;day^{-1}})$, so a bigger number is a *more potent* compound, as with pIC$_{50}$;
* the features are **mechanistic** ($\alpha$-hydrogen counts, branching, $\beta$-substituents, ring size), computed with RDKit, next to a fingerprint model for comparison;
* the validation is **honest**: a random split *and* a cluster split, a mean-predictor baseline, a y-randomization test, an applicability domain, and a conformal prediction interval;
* Part I runs on a **seed set of about fifty nitrosamines** transcribed from the CPDB summary table (no downloads, runs in a minute). Part II recomputes the potencies from the **full CPDB files**, checks them against the seed values, and re-runs the pipeline on the larger set.

> **Not for regulatory use.** Acceptable-intake limits for nitrosamine drug-substance-related impurities are set by regulators using their own frameworks (read-across, the carcinogenic potency categorization approach, expert review). This notebook is a teaching model on a small, selected data set of positive rodent carcinogens. It will be wrong for chemistries it has not seen, and the applicability-domain cell exists to tell you when.

## 0. Setup

In [ ]:
import math, time, importlib.util, os, sys, subprocess, warnings, re, json, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
if importlib.util.find_spec("rdkit") is None:                       # RDKit is not preinstalled in Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")
import rdkit; print("RDKit", rdkit.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## 1. The seed data and the potency arithmetic
The seed set is the rat TD$_{50}$ (mg/kg body weight/day; harmonic mean over positive experiments where there are several) of 51 N-nitrosamines from the CPDB summary table: dialkyl, cyclic, aralkyl and functionalized ones, including hydroxy, oxo, allyl, trifluoro and heteroaromatic substituents. **Nitrosamides and nitrosoureas (direct-acting alkylators that need no activation) are deliberately left out**, and so are chemicals that were tested and found negative, because a TD$_{50}$ does not exist for them. The second choice matters: a model trained on positives only will tend to *overestimate* the potency of a weak or non-carcinogenic nitrosamine.

Because molecular weights differ a lot (74 g/mol for NDMA, 198 for N-nitrosodiphenylamine), the comparison that makes chemical sense is per mole, not per kilogram:
$$\mathrm{pTD}_{50}=-\log_{10}\frac{\mathrm{TD}_{50}[\mathrm{mg/kg/day}]}{1000\cdot M_w[\mathrm{g/mol}]}$$
**Stop and predict:** N-nitrosodimethylamine (NDMA) has a rat TD$_{50}$ of $0.0959$ mg/kg/day and N-nitrosodiphenylamine (NDPhA) $167$. In mg/kg the ratio is about $1700$. Is the ratio per mole larger or smaller, and by how much?

In [ ]:
def mol_weight(smi): return Descriptors.MolWt(Chem.MolFromSmiles(smi))
SEED = [
 # name, CAS, SMILES, rat TD50 (mg/kg/day) from the CPDB summary table
 ("N-Nitrosodimethylamine", "62-75-9", "CN(C)N=O", 0.0959), ("N-Nitrosodiethylamine", "55-18-5", "CCN(CC)N=O", 0.0265),
 ("N-Nitrosodipropylamine", "621-64-7", "CCCN(CCC)N=O", 0.186), ("N-Nitrosodibutylamine", "924-16-3", "CCCCN(CCCC)N=O", 0.691),
 ("N-Nitrosoethylmethylamine", "10595-95-6", "CCN(C)N=O", 0.0503), ("N-Nitrosopiperidine", "100-75-4", "O=NN1CCCCC1", 1.43),
 ("N-Nitrosopyrrolidine", "930-55-2", "O=NN1CCCC1", 0.799), ("N-Nitrosomorpholine", "59-89-2", "O=NN1CCOCC1", 0.109),
 ("N-Nitrosodiethanolamine", "1116-54-7", "OCCN(CCO)N=O", 3.17), ("N-Nitrosodiphenylamine", "86-30-6", "O=NN(c1ccccc1)c1ccccc1", 167.0),
 ("Nitrosoheptamethyleneimine", "20917-49-1", "O=NN1CCCCCCC1", 0.0378), ("Nitrosododecamethyleneimine", "40580-89-0", "O=NN1CCCCCCCCCCCC1", 10.9),
 ("N-Nitrosothiomorpholine", "26541-51-5", "O=NN1CCSCC1", 5.39), ("Dinitrosohomopiperazine", "55557-00-1", "O=NN1CCCN(N=O)CC1", 0.0615),
 ("Nitroso-1,2,3,6-tetrahydropyridine", "55556-92-8", "O=NN1CCC=CC1", 0.0601), ("Nitrosoanabasine", "1133-64-8", "O=NN1CCCCC1c1cccnc1", 11.9),
 ("N'-Nitrosonornicotine", "53759-22-1", "O=NN1CCCC1c1cccnc1", 0.0957), ("NNK", "64091-91-4", "CN(CCCC(=O)c1cccnc1)N=O", 0.0999),
 ("NNAL", "76014-81-8", "CN(CCCC(O)c1cccnc1)N=O", 0.103), ("Dipentylnitrosamine", "13256-06-9", "CCCCCN(CCCCC)N=O", 4.03),
 ("Diallylnitrosamine", "16338-97-9", "C=CCN(CC=C)N=O", 33.9), ("N-Nitrosobis(2-hydroxypropyl)amine", "53609-64-6", "CC(O)CN(CC(C)O)N=O", 0.846),
 ("N-Nitrosobis(2-oxopropyl)amine", "60599-38-4", "CC(=O)CN(CC(C)=O)N=O", 0.491), ("N-Nitrosomethyl-2-hydroxypropylamine", "75411-83-5", "CC(O)CN(C)N=O", 0.0463),
 ("N-Nitrosomethyl(2-oxopropyl)amine", "55984-51-5", "CC(=O)CN(C)N=O", 0.0172), ("N-Nitrosomethyl(2-hydroxyethyl)amine", "26921-68-6", "CN(CCO)N=O", 1.29),
 ("N-Nitrosomethyl(3-hydroxypropyl)amine", "70415-59-7", "CN(CCCO)N=O", 1.66), ("N-Nitrosomethyldecylamine", "75881-22-0", "C" + "C" * 9 + "N(C)N=O", 1.26),
 ("Nitrosomethylundecylamine", "68107-26-6", "C" + "C" * 10 + "N(C)N=O", 2.37), ("N-Nitroso-N-methyl-N-dodecylamine", "55090-44-3", "C" + "C" * 11 + "N(C)N=O", 0.537),
 ("N-Nitroso-N-methyl-N-tetradecylamine", "75881-20-8", "C" + "C" * 13 + "N(C)N=O", 1.65), ("Nitrosomethylaniline", "614-00-6", "CN(N=O)c1ccccc1", 0.142),
 ("Nitroso-N-methyl-N-(2-phenyl)ethylamine", "13256-11-6", "CN(CCc1ccccc1)N=O", 0.00998), ("N-Nitroso-N-methyl-4-fluoroaniline", "937-25-7", "CN(N=O)c1ccc(F)cc1", 0.255),
 ("2-Nitrosomethylaminopyridine", "16219-98-0", "CN(N=O)c1ccccn1", 0.214), ("N-Nitroso-2,3-dihydroxypropyl-2-hydroxypropylamine", "89911-79-5", "CC(O)CN(CC(O)CO)N=O", 0.0535),
 ("Nitroso-2,3-dihydroxypropyl-2-oxopropylamine", "92177-50-9", "CC(=O)CN(CC(O)CO)N=O", 0.0352), ("N-Nitrosomethyl-2,3-dihydroxypropylamine", "86451-37-8", "CN(CC(O)CO)N=O", 0.646),
 ("N-Nitrosoallyl-2-oxopropylamine", "91308-71-3", "C=CCN(CC(C)=O)N=O", 0.335), ("N-Nitrosoallylethanolamine", "91308-69-9", "C=CCN(CCO)N=O", 0.491),
 ("N-Nitrosoallyl-2-hydroxypropylamine", "91308-70-2", "C=CCN(CC(C)O)N=O", 0.877), ("Nitroso-2-oxopropylethanolamine", "92177-49-6", "CC(=O)CN(CCO)N=O", 1.8),
 ("N-Nitroso(2-hydroxypropyl)(2-hydroxyethyl)amine", "75896-33-2", "CC(O)CN(CCO)N=O", 1.02), ("Nitrosomethyl-3-carboxypropylamine", "61445-55-4", "CN(CCCC(O)=O)N=O", 0.982),
 ("N-Nitroso(2,2,2-trifluoroethyl)ethylamine", "82018-90-4", "CCN(CC(F)(F)F)N=O", 2.52), ("N-Nitrosobis(4,4,4-trifluorobutyl)amine", "83335-32-4", "FC(F)(F)CCCN(CCCC(F)(F)F)N=O", 0.748),
 ("N-Nitrosoephedrine", "17608-59-2", "CC(C(O)c1ccccc1)N(C)N=O", 95.2), ("1-Nitroso-3,5-dimethyl-4-benzoylpiperazine", "61034-40-0", "CC1CN(N=O)CC(C)N1C(=O)c1ccccc1", 9.66),
 ("1-Nitroso-3,4,5-trimethylpiperazine", "75881-18-4", "CC1CN(N=O)CC(C)N1C", 0.151), ("N-Nitroso-3-hydroxypyrrolidine", "56222-35-6", "OC1CCN(N=O)C1", 7.65),
 ("N-Nitrosomethyl(2-tosyloxyethyl)amine", "", "CN(CCOS(=O)(=O)c1ccc(C)cc1)N=O", 4.8)]
seed = pd.DataFrame(SEED, columns=["name", "cas", "smiles", "td50_mg"])
seed["mw"] = seed.smiles.map(mol_weight)
to_ptd50 = lambda td50_mg, mw: -np.log10(td50_mg / (1000.0 * mw)); from_ptd50 = lambda p, mw: 10 ** (-p) * 1000.0 * mw
seed["ptd50"] = to_ptd50(seed.td50_mg, seed.mw)
print(f"{len(seed)} nitrosamines; rat TD50 from {seed.td50_mg.min():.4f} to {seed.td50_mg.max():.0f} mg/kg/day ({math.log10(seed.td50_mg.max() / seed.td50_mg.min()):.1f} orders of magnitude); pTD50 from {seed.ptd50.min():.2f} to {seed.ptd50.max():.2f}")
a, b = seed.set_index("name").loc["N-Nitrosodimethylamine"], seed.set_index("name").loc["N-Nitrosodiphenylamine"]
print(f"NDMA pTD50 {a.ptd50:.3f} (MW {a.mw:.1f}); NDPhA pTD50 {b.ptd50:.3f} (MW {b.mw:.1f}); ratio per kg {b.td50_mg / a.td50_mg:.0f}x, per mole {10 ** (a.ptd50 - b.ptd50):.0f}x")
fig, ax = plt.subplots(figsize=(6.2, 2.6)); ax.hist(seed.ptd50, bins=14, color=COLORS["blue"], alpha=.85); ax.set_xlabel(r"pTD$_{50}$ (rat, molar; larger = more potent)"); ax.set_ylabel("nitrosamines"); plt.tight_layout(); plt.show()
print(seed.sort_values("ptd50", ascending=False)[["name", "td50_mg", "ptd50"]].head(5).round(3).to_string(index=False)); print("..."); print(seed.sort_values("ptd50", ascending=False)[["name", "td50_mg", "ptd50"]].tail(4).round(3).to_string(index=False))

## 2. From mechanism to features
For each nitrosamine we locate the group $\mathrm{R_1R_2N{-}N{=}O}$ (excluding nitrosamides, where one substituent is an acyl group), and describe the two $\alpha$-carbons, the carbons directly bonded to the nitrogen. To make the description independent of which substituent is written first, the two sides are **sorted by their number of $\alpha$-hydrogens**: `aH_lo` is the weaker side, `aH_hi` the stronger one. The features, with the chemistry behind them:

| feature | what it counts | why it should matter |
|---|---|---|
| `aH_lo`, `aH_hi`, `aH_tot` | hydrogens on the $\alpha$-carbons | no $\alpha$-H, no $\alpha$-hydroxylation |
| `n_aCH3`, `n_aCH2`, `n_aCH`, `n_aC0` | $\alpha$-carbons that are methyl, methylene, methine, or have no H (aryl or quaternary) | methyl and methylene are the easiest to hydroxylate; a methine ($\alpha$-branching) is hindered |
| `br_lo`, `br_hi` | heavy-atom neighbours of each $\alpha$-carbon besides N | steric hindrance at the site of attack |
| `beta_OH`, `beta_oxo`, `beta_F`, `vinyl` | hydroxyl, carbonyl, fluorine or C=C on the carbon next to the $\alpha$-carbon | electronic effects on the $\alpha$-C-H bond, and metabolic pathways of their own |
| `arom_sides` | $\alpha$-carbons that are aromatic | an aryl nitrogen substituent cannot be hydroxylated at that carbon |
| `ring`, `ring_size`, `ring_het` | nitrogen in a ring, the ring size, other heteroatoms in the ring | cyclic nitrosamines behave differently (NMOR against NPIP) |
| `size_lo`, `size_hi`, `MW`, `logP`, `TPSA`, `nHeavy`, `nRot`, `fsp3`, `nArom`, `nHal` | size and polarity | transport, solubility, chain length |

**Stop and predict:** how many $\alpha$-hydrogens do NDMA, NDEA and NDPhA have in total?

In [ ]:
NITROSAMINE = Chem.MolFromSmarts("[#6][NX3;!$(N-[#6]=[O,S]);!$(N-S(=O)=O)](-[#6])-N=O")      # nitrosamine, not nitrosamide
def branch_size(mol, a_idx, n_idx):
    seen, stack, cnt = {n_idx}, [a_idx], 0
    while stack:
        i = stack.pop()
        if i in seen: continue
        seen.add(i); cnt += 1; stack += [x.GetIdx() for x in mol.GetAtomWithIdx(i).GetNeighbors() if x.GetIdx() not in seen]
    return cnt
def side_features(mol, a_idx, n_idx):
    a = mol.GetAtomWithIdx(a_idx); arom = a.GetIsAromatic(); oh = oxo = nF = vinyl = aO = 0
    for b_ in a.GetNeighbors():
        if b_.GetIdx() == n_idx: continue
        if b_.GetSymbol() == "O": aO += 1; continue
        if b_.GetSymbol() != "C": continue
        for g in b_.GetNeighbors():
            if g.GetIdx() == a_idx: continue
            order = mol.GetBondBetweenAtoms(b_.GetIdx(), g.GetIdx()).GetBondTypeAsDouble()
            if g.GetSymbol() == "O": oxo += order == 2; oh += (order == 1 and g.GetTotalNumHs() > 0)
            elif g.GetSymbol() == "F": nF += 1
            elif g.GetSymbol() == "C" and order == 2: vinyl += 1
    return dict(aH=0 if arom else a.GetTotalNumHs(), br=a.GetDegree() - 1, arom=int(arom), oh=int(oh), oxo=int(oxo), nF=nF, vinyl=vinyl, aO=aO, size=branch_size(mol, a_idx, n_idx))
def featurize_one(smi):
    mol = Chem.MolFromSmiles(smi); m = mol.GetSubstructMatches(NITROSAMINE) if mol is not None else ()
    if not m: return None
    c1, n, c2 = m[0][0], m[0][1], m[0][2]
    lo, hi = sorted([side_features(mol, c1, n), side_features(mol, c2, n)], key=lambda s: (s["aH"], s["br"], s["size"]))
    ri = mol.GetRingInfo(); ring = min((r for r in ri.AtomRings() if n in r), key=len, default=())
    aHs = [lo["aH"], hi["aH"]]
    return dict(aH_lo=lo["aH"], aH_hi=hi["aH"], aH_tot=sum(aHs), n_aCH3=aHs.count(3), n_aCH2=aHs.count(2), n_aCH=aHs.count(1), n_aC0=aHs.count(0),
                br_lo=lo["br"], br_hi=hi["br"], beta_OH=lo["oh"] + hi["oh"], beta_oxo=lo["oxo"] + hi["oxo"], beta_F=lo["nF"] + hi["nF"], vinyl=lo["vinyl"] + hi["vinyl"],
                alpha_O=lo["aO"] + hi["aO"], arom_sides=lo["arom"] + hi["arom"], ring=int(len(ring) > 0), ring_size=len(ring),
                ring_het=(sum(mol.GetAtomWithIdx(i).GetSymbol() != "C" for i in ring) - 1) if ring else 0, size_lo=lo["size"], size_hi=hi["size"],
                MW=Descriptors.MolWt(mol), logP=Crippen.MolLogP(mol), TPSA=rdMolDescriptors.CalcTPSA(mol), nHeavy=mol.GetNumHeavyAtoms(),
                nRot=rdMolDescriptors.CalcNumRotatableBonds(mol), fsp3=rdMolDescriptors.CalcFractionCSP3(mol), nArom=rdMolDescriptors.CalcNumAromaticRings(mol),
                nHal=sum(a.GetSymbol() in ("F", "Cl", "Br", "I") for a in mol.GetAtoms()))
def featurize(smiles):
    rows = [featurize_one(s) for s in smiles]; ok = [r is not None for r in rows]
    return pd.DataFrame([r for r in rows if r is not None]), np.array(ok)
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
def fp_counts(smiles): return np.array([_gen.GetCountFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in smiles], float)
def tanimoto(A, B):
    """count-vector (Ruzicka) Tanimoto similarity between every row of A and every row of B"""
    out = np.zeros((len(A), len(B)))
    for i in range(len(A)): out[i] = np.minimum(A[i], B).sum(1) / np.maximum(np.maximum(A[i], B).sum(1), 1e-9)
    return out
def leader_groups(sim, thr=0.5):
    """greedy leader clustering: groups of molecules with similarity >= thr to the group's leader"""
    n = len(sim); g = -np.ones(n, int); order = np.argsort(-(sim >= thr).sum(1)); k = 0
    for i in order:
        if g[i] >= 0: continue
        members = np.where((sim[i] >= thr) & (g < 0))[0]; g[members] = k; k += 1
    return g
F, ok = featurize(seed.smiles); assert ok.all(), "a seed structure was not recognized as a nitrosamine"
FEATURES = list(F.columns); Xm = F.values.astype(float); y = seed.ptd50.values; Xf = fp_counts(seed.smiles); Xb = np.hstack([Xm, Xf])
print(len(FEATURES), "mechanistic features;", Xf.shape[1], "fingerprint bits")
show = seed[["name"]].join(F[["aH_lo", "aH_hi", "aH_tot", "br_lo", "br_hi", "beta_OH", "beta_oxo", "ring", "ring_size"]]).set_index("name")
print(show.loc[["N-Nitrosodimethylamine", "N-Nitrosodiethylamine", "N-Nitrosodiphenylamine", "Nitrosomethylaniline", "N-Nitrosopiperidine", "N-Nitrosobis(2-hydroxypropyl)amine"]].to_string())
print("\ncorrelation of each feature with pTD50 (Spearman):"); from scipy.stats import spearmanr
cor = pd.Series({c: spearmanr(F[c], y)[0] for c in FEATURES if F[c].nunique() > 1}).sort_values(); print(cor.round(2).to_string())

## 3. Models, and two ways to validate them
Seven models and a baseline, on three feature sets: the mechanistic features (`mech`), a Morgan count fingerprint of 1024 bits (`fp`), and both (`mech+fp`). The baseline predicts the training mean for everybody; **any model that cannot beat it has learned nothing**.

Two validation schemes answer different questions.

* **Random 5-fold cross-validation** (repeated five times) asks: *how well can the model interpolate among compounds similar to the ones it has seen?* Series such as the N-nitroso-N-methyl-alkylamines put close neighbours in both training and test folds, so this is the optimistic number.
* **Cluster 5-fold cross-validation** puts whole clusters of similar molecules (leader clustering at Tanimoto 0.5) into the test fold, so every test compound is at least a bit unfamiliar. This is closer to the situation of a new impurity that is not in the database, and it is the number to believe.

The headline metric is $Q^2=1-\sum(y_i-\hat y_i)^2/\sum(y_i-\bar y)^2$ on out-of-fold predictions (1 is perfect, 0 is no better than the mean, negative is worse). We also report the RMSE in log$_{10}$ units (an RMSE of $0.5$ is a typical error of $3.2$-fold) and the fraction of compounds predicted within 3-fold and within 10-fold.

**Stop and predict:** which scheme will give the larger $Q^2$, and which feature set will degrade most when you move from the random to the cluster split?

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import KFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
SETS = {"mech": Xm, "fp": Xf, "mech+fp": Xb}
ZOO = {"mean baseline": (DummyRegressor(), "mech"),
       "ridge | mech": (make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))), "mech"),
       "svr | mech": (make_pipeline(StandardScaler(), SVR(C=3.0, epsilon=0.1)), "mech"),
       "knn(3) | mech": (make_pipeline(StandardScaler(), KNeighborsRegressor(3, weights="distance")), "mech"),
       "rf | mech": (RandomForestRegressor(300, min_samples_leaf=2, random_state=0, n_jobs=-1), "mech"),
       "gbm | mech": (GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=0), "mech"),
       "rf | fp": (RandomForestRegressor(300, min_samples_leaf=2, random_state=0, n_jobs=-1), "fp"),
       "rf | mech+fp": (RandomForestRegressor(300, min_samples_leaf=2, random_state=0, n_jobs=-1), "mech+fp")}
def metrics(y_, p_):
    r = y_ - p_; return dict(Q2=1 - (r ** 2).sum() / ((y_ - y_.mean()) ** 2).sum(), RMSE=math.sqrt((r ** 2).mean()), MAE=np.abs(r).mean(),
                             rho=spearmanr(y_, p_)[0] if np.std(p_) > 0 else np.nan, within3=np.mean(np.abs(r) < math.log10(3)), within10=np.mean(np.abs(r) < 1.0))
def oof_random(est, X, y_, reps=5, k=5):
    P = np.zeros((reps, len(y_)))
    for r in range(reps):
        for tr, te in KFold(k, shuffle=True, random_state=r).split(X): P[r, te] = clone(est).fit(X[tr], y_[tr]).predict(X[te])
    return P
def oof_group(est, X, y_, groups, k=5):
    P = np.zeros(len(y_))
    for tr, te in GroupKFold(k).split(X, y_, groups): P[te] = clone(est).fit(X[tr], y_[tr]).predict(X[te])
    return P
S_train = tanimoto(Xf, Xf); thr = 0.5; groups = leader_groups(S_train, thr)
while len(set(groups)) < 6 and thr > 0.2: thr -= 0.1; groups = leader_groups(S_train, thr)
print(f"{len(set(groups))} clusters at Tanimoto >= {thr:.1f}; largest cluster has {np.bincount(groups).max()} molecules")
t0 = time.perf_counter(); rows, OOF_R, OOF_G = [], {}, {}
for name, (est, fs) in ZOO.items():
    X = SETS[fs]; PR = oof_random(est, X, y); PG = oof_group(est, X, y, groups); OOF_R[name], OOF_G[name] = PR, PG
    mr = pd.DataFrame([metrics(y, p) for p in PR]).mean(); mg = metrics(y, PG)
    rows.append(dict(model=name, Q2_random=mr.Q2, RMSE_random=mr.RMSE, Q2_cluster=mg["Q2"], RMSE_cluster=mg["RMSE"], within3_cluster=mg["within3"], within10_cluster=mg["within10"], rho_cluster=mg["rho"]))
res = pd.DataFrame(rows).set_index("model"); print(res.round(2).to_string()); print(f"({time.perf_counter() - t0:.0f} s)")
cands = [m for m in res.index if m != "mean baseline"]; BEST = res.loc[cands, "RMSE_cluster"].idxmin()
print(f"\nbest model by cluster-split RMSE: {BEST}  (Q2 {res.loc[BEST, 'Q2_cluster']:.2f}, RMSE {res.loc[BEST, 'RMSE_cluster']:.2f} log units = {10 ** res.loc[BEST, 'RMSE_cluster']:.1f}-fold)")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.8), sharex=True, sharey=True); lim = [y.min() - .4, y.max() + .4]
for a_, (ttl, p_) in zip(ax, [("random 5-fold CV (mean of 5 repeats)", OOF_R[BEST].mean(0)), ("cluster 5-fold CV", OOF_G[BEST])]):
    a_.scatter(y, p_, s=22, color=COLORS["blue"], alpha=.85); a_.plot(lim, lim, "k--", lw=.8); a_.fill_between(lim, [l - 1 for l in lim], [l + 1 for l in lim], color=COLORS["gray"], alpha=.12)
    a_.set_title(f"{BEST}: {ttl}\nQ2 {metrics(y, p_)['Q2']:.2f}, RMSE {metrics(y, p_)['RMSE']:.2f}", fontsize=9); a_.set_xlabel(r"experimental pTD$_{50}$")
ax[0].set_ylabel(r"predicted pTD$_{50}$ (out-of-fold)"); ax[0].set_xlim(lim); ax[0].set_ylim(lim); plt.tight_layout(); plt.show()

## 4. Is the model learning anything? y-randomization and a mechanistic baseline
Two cheap tests protect against self-deception on a data set this small.

**y-randomization.** Shuffle the potencies among the molecules and repeat the whole cross-validation. If a shuffled target scores almost as well as the real one, the model is fitting noise. The $p$-value is the fraction of shuffles whose $Q^2$ reaches the real one.

**A one-number mechanistic baseline.** Predict from the total number of $\alpha$-hydrogens alone (`aH_tot`) with a straight line. If the full model cannot beat that, the extra features are not earning their keep.

**Stop and predict:** will the $\alpha$-hydrogen count alone explain more or less than a third of the variance?

In [ ]:
from sklearn.linear_model import LinearRegression
est_best, fs_best = ZOO[BEST]; Xb_ = SETS[fs_best]; rg = np.random.default_rng(0); n_perm = 60 if BEST.startswith(("ridge", "svr", "knn")) else 30
def q2_random(est, X, y_, seed=0):
    P = np.zeros(len(y_))
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(X): P[te] = clone(est).fit(X[tr], y_[tr]).predict(X[te])
    return metrics(y_, P)["Q2"]
t0 = time.perf_counter(); q_true = q2_random(est_best, Xb_, y); q_perm = np.array([q2_random(est_best, Xb_, rg.permutation(y)) for _ in range(n_perm)])
p_val = (1 + (q_perm >= q_true).sum()) / (1 + n_perm)
print(f"{BEST}: real Q2 = {q_true:.2f}; {n_perm} shuffled targets: mean Q2 = {q_perm.mean():.2f}, max = {q_perm.max():.2f}; p = {p_val:.3f}  ({time.perf_counter() - t0:.0f} s)")
aH = Xm[:, FEATURES.index("aH_tot")][:, None]; base_aH = LinearRegression(); P_aH = oof_group(base_aH, aH, y, groups); m_aH = metrics(y, P_aH)
print(f"alpha-hydrogen count alone (cluster CV): Q2 = {m_aH['Q2']:.2f}, RMSE = {m_aH['RMSE']:.2f};   full model: Q2 = {res.loc[BEST, 'Q2_cluster']:.2f}, RMSE = {res.loc[BEST, 'RMSE_cluster']:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.2)); ax[0].hist(q_perm, bins=15, color=COLORS["gray"], alpha=.8, label="shuffled targets"); ax[0].axvline(q_true, color=COLORS["red"], lw=2, label="real target"); ax[0].set_xlabel(r"$Q^2$ (random 5-fold)"); ax[0].legend(frameon=False)
ax[1].scatter(F.aH_tot + rg.normal(0, .06, len(F)), y, s=22, color=COLORS["blue"], alpha=.8); ax[1].set_xlabel(r"total $\alpha$-hydrogens (jittered)"); ax[1].set_ylabel(r"pTD$_{50}$"); plt.tight_layout(); plt.show()
if hasattr(est_best, "steps") and isinstance(est_best.steps[-1][1], RidgeCV):        # ridge: show what it learned
    r_ = clone(est_best).fit(Xm, y); coef = pd.Series(r_.steps[-1][1].coef_, index=FEATURES).sort_values(); print("\nridge coefficients on standardized features (pTD50 change per +1 s.d.):"); print(pd.concat([coef.head(5), coef.tail(5)]).round(2).to_string())
else:
    from sklearn.inspection import permutation_importance
    r_ = clone(est_best).fit(Xb_, y); bits_ = [f"bit{i}" for i in range(Xf.shape[1])]; names_ = {"mech": FEATURES, "fp": bits_, "mech+fp": FEATURES + bits_}[fs_best]
    pi = permutation_importance(r_, Xb_, y, n_repeats=10, random_state=0); imp = pd.Series(pi.importances_mean, index=names_).sort_values(ascending=False); print("\npermutation importance (training data, top 8):"); print(imp.head(8).round(3).to_string())

## 5. When to trust a prediction: applicability domain and a prediction interval
A model should say not only *what* it predicts but *whether it should be asked*. Three checks, all cheap.

1. **Similarity.** The largest Tanimoto similarity between the query and any training molecule. Prediction error grows as this falls; the plot below shows the relation in our own leave-one-out data.
2. **Mechanistic coverage.** The model has seen nitrosamines with $\alpha$-hydrogens on at least one side and a molecular weight and logP in a certain range. Outside it, the model is extrapolating. A nitrosamine with *no* $\alpha$-hydrogens at all is outside the chemistry this model has been taught (only N-nitrosodiphenylamine is like that in the seed set).
3. **A prediction interval.** Split-conformal: take the absolute out-of-fold errors from the cluster split, and use their 90th percentile $q_{90}$ (with the finite-sample correction) as a $\pm q_{90}$ half-width around every prediction. The guarantee is only as good as the assumption that new compounds resemble the cluster-split test compounds, and we check coverage with leave-one-out versions of the same interval.

**Stop and predict:** with 51 compounds, will the 90% interval be narrower or wider than a factor of 10 in dose?

In [ ]:
resid = np.abs(y - OOF_G[BEST]); n_ = len(resid); level = 0.90
qhat = np.quantile(resid, min(1.0, math.ceil((n_ + 1) * level) / n_), method="higher")
cov = np.mean([resid[i] <= np.quantile(np.delete(resid, i), min(1.0, math.ceil(n_ * level) / (n_ - 1)), method="higher") for i in range(n_)])
print(f"conformal half-width q90 = {qhat:.2f} log10 units  ->  the 90% interval is a factor of {10 ** qhat:.1f} either way ({10 ** (2 * qhat):.0f}-fold wide);  leave-one-out coverage {cov:.2f}")
S_off = S_train.copy(); np.fill_diagonal(S_off, 0); sim_loo = S_off.max(1)
rho_ad = spearmanr(sim_loo, resid)[0]; print(f"leave-one-out: Spearman(max similarity, |error|) = {rho_ad:.2f}")
fig, ax = plt.subplots(figsize=(5.4, 3.3)); ax.scatter(sim_loo, resid, s=22, color=COLORS["blue"], alpha=.8); ax.axhline(qhat, color=COLORS["red"], ls="--", lw=1); ax.text(sim_loo.min(), qhat + .03, "90% half-width", color=COLORS["red"], fontsize=8)
ax.set_xlabel("max Tanimoto similarity to the rest of the training set"); ax.set_ylabel(r"|error| (cluster CV, log$_{10}$)"); plt.tight_layout(); plt.show()
FINAL = clone(est_best).fit(SETS[fs_best], y); dom = dict(MW=(F.MW.min(), F.MW.max()), logP=(F.logP.min(), F.logP.max()), TPSA=(F.TPSA.min(), F.TPSA.max()))
def predict_smiles(smiles, min_sim=0.35):
    rows = []
    for s in smiles:
        f = featurize_one(s)
        if f is None: rows.append(dict(smiles=s, note="not an N-nitrosamine (or not parseable)")); continue
        fm = pd.DataFrame([f])[FEATURES].values.astype(float); ff = fp_counts([s]); X_ = {"mech": fm, "fp": ff, "mech+fp": np.hstack([fm, ff])}[fs_best]
        p = float(FINAL.predict(X_)[0]); sim = float(tanimoto(ff, Xf).max()); mw = f["MW"]; flags = []
        if f["aH_tot"] == 0: flags.append("no alpha-H: outside the mechanism this model was trained on")
        if sim < min_sim: flags.append(f"low similarity ({sim:.2f})")
        for k_, v_ in dom.items():
            if not dom[k_][0] <= f[k_] <= dom[k_][1]: flags.append(f"{k_} outside the training range")
        if s in set(seed.smiles): flags.append("in the training set")
        rows.append(dict(smiles=s, aH_lo=f["aH_lo"], aH_hi=f["aH_hi"], pTD50=p, lo90=p - qhat, hi90=p + qhat, TD50_mg_kg_day=from_ptd50(p, mw), TD50_low=from_ptd50(p + qhat, mw), TD50_high=from_ptd50(p - qhat, mw), max_sim=sim, in_domain=not [x for x in flags if not x.startswith("in the training")], note="; ".join(flags)))
    return pd.DataFrame(rows)

## 6. Using the model
We predict a few nitrosamines that are **not in the seed set** (plus one that is, as a sanity check). Read the `note` column before the numbers. Several of these are chosen to probe the mechanism: an $\alpha$-branched nitrosamine (isopropyl), a *tert*-butyl one (no $\alpha$-hydrogen on that side), and a nitrosamine with no $\alpha$-hydrogens at all.

**Stop and predict:** N-nitroso-di-*tert*-butylamine has no $\alpha$-hydrogens on either side. A purely statistical model with no mechanistic features would predict something near the average potency. What *should* the prediction be, mechanistically, and what does our table do with it?

The predicted TD$_{50}$ here is a point estimate for **rat, chronic dosing, positive carcinogens**. It is not an acceptable-intake limit.

In [ ]:
EXAMPLES = {"N-nitrosomethylbutylamine": "CCCCN(C)N=O", "1-methyl-4-nitrosopiperazine": "O=NN1CCN(C)CC1", "N-nitrosomethylisopropylamine": "CC(C)N(C)N=O",
            "N-nitrosodiisopropylamine": "CC(C)N(C(C)C)N=O", "N-nitrosomethyl-tert-butylamine": "CC(C)(C)N(C)N=O", "N-nitroso-di-tert-butylamine": "CC(C)(C)N(C(C)(C)C)N=O",
            "N-nitrosomethylcyclohexylamine": "O=NN(C)C1CCCCC1", "N-nitrosomethylbenzylamine": "O=NN(C)Cc1ccccc1", "N-nitrosomorpholine (in training)": "O=NN1CCOCC1"}
pred = predict_smiles(list(EXAMPLES.values())); pred.insert(0, "name", list(EXAMPLES))
print(pred[["name", "aH_lo", "aH_hi", "pTD50", "TD50_mg_kg_day", "TD50_low", "TD50_high", "max_sim", "in_domain"]].round(3).to_string(index=False)); print()
for _, r in pred.iterrows():
    if r.get("note"): print(f"{r['name']}: {r['note']}")

## 7. Your turn
**Exercise 1.** Remove the mechanistic features one group at a time (the $\alpha$-hydrogen block, then the $\beta$-substituents, then the size and polarity block) and re-run the cluster CV. Which group does the model miss most?

**Exercise 2.** Change the clustering threshold from $0.5$ to $0.3$ and to $0.7$. How does $Q^2_\mathrm{cluster}$ move, and why is the choice of threshold part of the result and not a detail?

**Exercise 3.** Model $\log_{10}\mathrm{TD}_{50}$ in mg/kg/day instead of the molar $\mathrm{pTD}_{50}$. Do the rankings of the models change? Does the molecular-weight column become more or less important?

**Exercise 4.** Add a sign-aware feature of your own, for example "has a CH$_2$ at the $\alpha$-position *and* a hydroxyl or ether at the $\beta$-position", and check with a paired comparison of cluster-CV errors whether it helps.

**Exercise 5.** The seed data contain only positives. Add five *fictitious* weak or non-carcinogenic nitrosamines (say, pTD$_{50}$ around $2$) with no $\alpha$-hydrogens and see how the model, the applicability domain and the interval change. What does this say about the value of real negative data?

## 8. Self-check

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
fx = lambda smi: featurize_one(smi)
check("NDMA has 6 alpha-H (3+3)", fx("CN(C)N=O")["aH_tot"] == 6 and fx("CN(C)N=O")["n_aCH3"] == 2)
check("NDEA has 4 alpha-H (2+2), both methylene", fx("CCN(CC)N=O")["aH_tot"] == 4 and fx("CCN(CC)N=O")["n_aCH2"] == 2)
check("N-nitrosodiphenylamine has no alpha-H and two aromatic sides", fx("O=NN(c1ccccc1)c1ccccc1")["aH_tot"] == 0 and fx("O=NN(c1ccccc1)c1ccccc1")["arom_sides"] == 2)
check("N-nitrosomethylaniline: sorted sides (0, 3)", (fx("CN(N=O)c1ccccc1")["aH_lo"], fx("CN(N=O)c1ccccc1")["aH_hi"]) == (0, 3))
check("N-nitrosopiperidine: ring of 6, 4 alpha-H", fx("O=NN1CCCCC1")["ring_size"] == 6 and fx("O=NN1CCCCC1")["aH_tot"] == 4)
check("N-nitrosomorpholine: one extra ring heteroatom", fx("O=NN1CCOCC1")["ring_het"] == 1)
check("BOP has two beta-oxo groups; BHP has two beta-hydroxyls", fx("CC(=O)CN(CC(C)=O)N=O")["beta_oxo"] == 2 and fx("CC(O)CN(CC(C)O)N=O")["beta_OH"] == 2)
check("a nitrosamide is not matched", fx("CC(=O)N(C)N=O") is None)
check("feature vector is symmetric in the two substituents", fx("CCN(C)N=O")["aH_lo"] == fx("CN(CC)N=O")["aH_lo"] and fx("CCN(C)N=O")["size_hi"] == fx("CN(CC)N=O")["size_hi"])
check("NDMA: MW 74.08 and pTD50 5.888", abs(seed.set_index("name").loc["N-Nitrosodimethylamine", "mw"] - 74.08) < 0.02 and abs(seed.set_index("name").loc["N-Nitrosodimethylamine", "ptd50"] - 5.888) < 0.005)
check("pTD50 <-> TD50 round trip", np.allclose(from_ptd50(seed.ptd50.values, seed.mw.values), seed.td50_mg.values))
check("Tanimoto of a molecule with itself is 1", np.allclose(np.diag(S_train), 1.0))
check("the clusters are not all singletons nor a single block", 5 <= len(set(groups)) <= len(seed) - 1)
check("conformal leave-one-out coverage is near 90%", 0.75 <= cov <= 1.0)
check("di-tert-butyl nitrosamine is flagged as outside the domain", not bool(predict_smiles(["CC(C)(C)N(C(C)(C)C)N=O"]).in_domain.iloc[0]))
print(f"\ninformation (not pass/fail, depends on the data and the split): best {BEST} Q2 random {res.loc[BEST, 'Q2_random']:.2f}, cluster {res.loc[BEST, 'Q2_cluster']:.2f}; baseline Q2 cluster {res.loc['mean baseline', 'Q2_cluster']:.2f}; y-randomization p = {p_val:.3f}")
assert all(ok_ for _, ok_ in checks); print("All checks passed.")

# Part II. From the seed set to the full CPDB
> **Read this first.** Part II works with the **public CPDB data files**, which the notebook downloads from the Carcinogenic Potency Project's site (`files.toxplanet.com/cpdb/CPDB-tab.html`; the National Library of Medicine also distributes an archived copy under the Open Database License). It needs an internet connection and takes a few minutes. **(1)** The cell downloads the experiment files, and **recomputes the species-level TD$_{50}$** the way the CPDB summary table does (the harmonic mean, over the positive experiments of a chemical, of each experiment's most potent site). **(2)** It **checks the recomputation against the 51 seed values**; if the two agree for most chemicals, the parsing is right. **(3)** It looks up the structure of every chemical whose name contains "nitroso" (from PubChem by CAS number), keeps the N-nitrosamines, and **re-runs the Part I pipeline** on the bigger set. Every step is guarded: if a download or a lookup fails, the cell says so and Part I is unaffected.
>
> **What we do not know in advance.** The CPDB's `chemname` file has a column layout that we detect by name, and the NCI/NTP file uses the same columns as the literature file only as far as we can tell. The cell prints what it finds. If something is odd, read the printout before trusting the numbers.

In [ ]:
import urllib.request, io
BASE = "https://files.toxplanet.com/cpdb/tab/"; FILES = {"lit": "cpdb.lit.tab.txt", "ncintp": "cpdb.ncintp.tab.txt", "chemname": "cpdb.chemname.tab.txt"}
def fetch(name, retries=2):
    path = FILES[name]
    if os.path.isfile(path): return path
    for k in range(retries + 1):
        try: urllib.request.urlretrieve(BASE + path, path); return path
        except Exception as e_: print(f"download of {path} failed ({type(e_).__name__}: {e_})"); time.sleep(2)
    return None
def read_tab(path): return pd.read_csv(path, sep="\t", dtype=str, encoding="latin-1", on_bad_lines="skip", quotechar="\x00")
CPDB_OK = False
try:
    paths = {k: fetch(k) for k in FILES}
    if all(paths.values()):
        lit, nci, cn = read_tab(paths["lit"]), read_tab(paths["ncintp"]), read_tab(paths["chemname"])
        for d_ in (lit, nci, cn): d_.columns = [c.strip().lower() for c in d_.columns]
        print("literature file:", lit.shape, "| NCI/NTP file:", nci.shape, "| chemname file:", cn.shape); print("chemname columns:", cn.columns.tolist()); print(cn.head(3).to_string())
        common = [c for c in lit.columns if c in nci.columns]; print("columns shared by the two experiment files:", len(common), "of", len(lit.columns))
        exp = pd.concat([lit[common], nci[common]], ignore_index=True); CPDB_OK = True
    else: print("files missing: upload cpdb.lit.tab.txt, cpdb.ncintp.tab.txt and cpdb.chemname.tab.txt to the Colab file browser and re-run this cell")
except Exception as e_: print("CPDB download or parsing failed:", type(e_).__name__, e_)

In [ ]:
def species_td50(exp_, species="r"):
    """CPDB-style species harmonic mean of TD50: per experiment the most potent TD50 among sites with author's opinion '+', then the harmonic mean over experiments"""
    d_ = exp_[(exp_.species.str.strip().str.lower() == species) & (exp_.opinion.str.strip() == "+")].copy(); d_["td50"] = pd.to_numeric(d_.td50, errors="coerce")
    d_ = d_[d_.td50 < 1e27]                                                                            # 1E28 and 1E31 mean that no estimate exists
    d_["exp_id"] = d_[["papernum", "strain", "sex", "route", "xpotime", "xprtime"]].astype(str).agg("|".join, axis=1)
    best = d_.groupby(["chemcode", "exp_id"]).td50.min().reset_index()
    out = best.groupby("chemcode").td50.agg(n_exp="size", td50_hm=lambda v: len(v) / np.sum(1.0 / v)); out.index = out.index.str.strip(); return out
if CPDB_OK:
    exp["chemcode"] = exp.chemcode.str.strip(); rat_hm = species_td50(exp, "r"); print("chemicals with a rat harmonic-mean TD50 recomputed:", len(rat_hm))
    code_col = next(c for c in cn.columns if "code" in c); cas_col = next((c for c in cn.columns if "cas" in c), None); name_col = next((c for c in cn.columns if "name" in c and "code" not in c), None)
    cn["chemcode"] = cn[code_col].str.strip(); print(f"detected columns: code={code_col}, cas={cas_col}, name={name_col}")
    allc = cn.drop_duplicates("chemcode").set_index("chemcode").join(rat_hm, how="inner").reset_index()
    if cas_col is not None:
        chk = seed.merge(allc[[cas_col, "td50_hm", "n_exp"]].rename(columns={cas_col: "cas"}), on="cas", how="inner"); chk["log_ratio"] = np.log10(chk.td50_hm / chk.td50_mg)
        print(f"\n{len(chk)} seed nitrosamines found by CAS in the recomputation; median |log10 ratio| = {chk.log_ratio.abs().median():.3f}; within 10%: {np.mean(chk.log_ratio.abs() < 0.0414):.0%}; within 2-fold: {np.mean(chk.log_ratio.abs() < 0.301):.0%}")
        print(chk.loc[chk.log_ratio.abs() > 0.301, ["name", "td50_mg", "td50_hm", "n_exp"]].round(4).to_string(index=False) or "no discrepancy above 2-fold")
        fig, ax = plt.subplots(figsize=(4.4, 4.0)); ax.scatter(np.log10(chk.td50_mg), np.log10(chk.td50_hm), s=22, color=COLORS["green"]); lim = [-3, 3]; ax.plot(lim, lim, "k--", lw=.8)
        ax.set_xlabel("seed value from the CPDB summary table (log10 mg/kg/day)"); ax.set_ylabel("recomputed here from the experiment file"); plt.tight_layout(); plt.show()

In [ ]:
NAMEPAT = re.compile(r"nitroso", re.I); EXCL = re.compile(r"urea|urethan|guanidin|amide|carbam|nitrosoimid|hydantoin|uracil|oxazolidone|isothiocy|phenylhydrazine", re.I)
FULL = None
if CPDB_OK and cas_col is not None and name_col is not None:
    by_name = allc[name_col].fillna("").str.contains(NAMEPAT) & ~allc[name_col].fillna("").str.contains(EXCL); by_seed = allc[cas_col].isin(seed.cas)      # NNK and NNAL have no 'nitroso' in their names
    cand = allc[by_name | by_seed][[code_col, name_col, cas_col, "td50_hm", "n_exp"]].rename(columns={code_col: "code", name_col: "name", cas_col: "cas"})
    print(len(cand), "chemicals with 'nitroso' in the name (nitrosoureas, urethanes and similar excluded by name) or in the seed set, and a rat TD50")
    SMI_CACHE = "nitroso_smiles.json"; cache = json.load(open(SMI_CACHE)) if os.path.isfile(SMI_CACHE) else {}
    seed_smi = dict(zip(seed.cas, seed.smiles))
    try:
        if importlib.util.find_spec("pubchempy") is None: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pubchempy"])
        import pubchempy as pcp
    except Exception as e_: pcp = None; print("pubchempy not available:", e_)
    for cas in cand.cas.dropna().unique():
        if cas in seed_smi: cache[cas] = seed_smi[cas]
        elif cas not in cache and pcp is not None and re.match(r"^\d+-\d\d-\d$", cas):
            try: c_ = pcp.get_compounds(cas, "name"); cache[cas] = (c_[0].connectivity_smiles if c_ else None) or (c_[0].isomeric_smiles if c_ else None)
            except Exception as e_: cache[cas] = None
            time.sleep(0.25)
    json.dump(cache, open(SMI_CACHE, "w")); cand["smiles"] = cand.cas.map(cache); cand = cand.dropna(subset=["smiles"])
    Ff, okf = featurize(cand.smiles); cand = cand[okf].reset_index(drop=True); Ff = Ff.reset_index(drop=True)
    cand["mw"] = Ff.MW; cand["td50_mg"] = cand.td50_hm.astype(float); cand["ptd50"] = to_ptd50(cand.td50_mg, cand.mw); cand = cand.drop_duplicates("smiles").reset_index(drop=True)
    FULL = cand; print(f"{len(FULL)} N-nitrosamines with a structure and a rat TD50;  pTD50 from {FULL.ptd50.min():.2f} to {FULL.ptd50.max():.2f};  {FULL.smiles.isin(seed.smiles).sum()} of them were in the seed set, {(~FULL.smiles.isin(seed.smiles)).sum()} are new")
    print(FULL[~FULL.smiles.isin(seed.smiles)][["name", "cas", "td50_mg"]].head(12).round(4).to_string(index=False))
    FULL[["name", "cas", "smiles", "td50_mg"]].to_csv("full_cpdb_nitrosamines.csv", index=False); print("saved full_cpdb_nitrosamines.csv (used by the next notebook, B7o)")

In [ ]:
if FULL is not None and len(FULL) >= 30:
    FFull, _ = featurize(FULL.smiles); Xm2 = FFull[FEATURES].values.astype(float); Xf2 = fp_counts(FULL.smiles); y2 = FULL.ptd50.values; S2 = tanimoto(Xf2, Xf2)
    thr2 = 0.5; g2 = leader_groups(S2, thr2)
    while len(set(g2)) < 6 and thr2 > 0.2: thr2 -= 0.1; g2 = leader_groups(S2, thr2)
    SETS2 = {"mech": Xm2, "fp": Xf2, "mech+fp": np.hstack([Xm2, Xf2])}; rows2 = []; t0 = time.perf_counter()
    for name, (est, fs) in ZOO.items():
        PR = oof_random(est, SETS2[fs], y2); PG = oof_group(est, SETS2[fs], y2, g2); mr = pd.DataFrame([metrics(y2, p) for p in PR]).mean(); mg = metrics(y2, PG)
        rows2.append(dict(model=name, Q2_random=mr.Q2, RMSE_random=mr.RMSE, Q2_cluster=mg["Q2"], RMSE_cluster=mg["RMSE"], within3_cluster=mg["within3"], within10_cluster=mg["within10"]))
    res2 = pd.DataFrame(rows2).set_index("model"); print(f"n = {len(FULL)}; {len(set(g2))} clusters at Tanimoto >= {thr2:.1f}  ({time.perf_counter() - t0:.0f} s)"); print(res2.round(2).to_string())
    comp = pd.DataFrame({"seed (n=%d)" % len(seed): res.Q2_cluster, "full (n=%d)" % len(FULL): res2.Q2_cluster}); print("\ncluster-split Q2, seed against full:"); print(comp.round(2).to_string())
else: print("The full-CPDB nitrosamine set is not available in this session; Part I stands on its own.")

### What to look for
* **The recomputation check.** For most seed chemicals the recomputed harmonic mean should agree with the summary-table value to within a few percent, because both follow the same recipe. Chemicals that disagree by more than 2-fold are usually ones where the CPDB's choice of which experiments to pool, or its handling of NCI/NTP "inadequate" tests, differs from our simplified rule. The cell lists them; do not drop them silently, and check the original CPDB page of any chemical that matters to you.
* **Where the new compounds come from.** Names that contain "nitroso" and that PubChem resolves by CAS number. Chemicals that PubChem cannot resolve are dropped, so the full set is smaller than the CPDB's nitroso chemicals, and it is biased towards the well-documented ones.
* **How much more data helps.** Compare the cluster-split $Q^2$ in the two columns of the last table. If it does not improve much, the limit is not the sample size but the information in the features; look at the compounds with the largest errors and ask which mechanistic feature they would need.
* **What the model cannot do.** It was trained on potent rodent carcinogens only. It has no negatives, no species other than the rat, no information about human metabolism, and no knowledge of the very large nitrosamine drug-substance-related impurities, where steric hindrance and unusual substituents decide the potency. The applicability-domain flags and the prediction interval are there to say so. A regulatory acceptable-intake limit comes from experimental data, read-across to a close analogue, or the regulator's potency-categorization framework, with expert review, never from a notebook like this one.
* **Citations to check.** Gold *et al.*, *Environ. Health Perspect.* 1984; 58:9-319, and the later CPDB updates, for the database; Peto *et al.*, *Environ. Health Perspect.* 1984; 58:1-8, for the TD$_{50}$; the current FDA and EMA guidance documents on nitrosamine impurities for the regulatory framework (read the version in force, since it changes).